In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "haun2011great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "casino_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="haun2011great"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

df.rename(columns={"subject": "ape",
    "species":"species_temp",
    "sex":"sex_temp"}, inplace=True)


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [4]:
df[['month','day', 'year']] = df['date'].str.split('/',expand=True)
df['year'] = '20' + df['year'].astype(str)


In [5]:
code_list=["condition"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '0':
            entry = "visible"
        elif entry =='1':
            entry = "invisible"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [6]:
code_list=["success"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '0':
            entry = "no"
        elif entry =='1':
            entry = "yes"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [7]:
code_list=["first"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '0':
            entry = "small_first"
        elif entry =='1':
            entry = "big_first"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [8]:
code_list=["risk choice"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '1':
            entry = "risk"
        elif entry =='0':
            entry = "safe"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [9]:

code_list=["ordervar1"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '0':
            entry = "choice_x_not_choice_x_plus_1"
        elif entry =='1':
            entry = "choice_x=choice_x_plus_1"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [10]:
import re
replace_1=re.compile('(\.|\ )') 
# replace_2=re.compile('( |-|\(|\))') # ' ', -, ( and )
df.columns = df.columns.str.replace(replace_1, '_')
df.columns = df.columns.str.replace('__', '_')
df['ordervar1'].replace('nan', '', inplace=True)
df['ordervar1_codes'].replace('nan', '', inplace=True)


In [11]:
df['what_cups'].replace('\,', '_', inplace=True, regex=True)
df['size'].replace('\,', '.', inplace=True, regex=True)
df.rename(columns={"ape": "participant"}, inplace=True)
# df.columns

update_cup_list = [['4','5'],
                    ['3','4'],
                    ['2','3'],
                    ['1','2']]
for x,y in update_cup_list:
    df['what_cups'].replace(x, y, inplace=True, regex=True)

update_cup_list_int = [[4,5],
                    [3,4],
                    [2,3],
                    [1,2]] 
for x,y in update_cup_list_int:
    df['baited_cup'].replace(x, y, inplace=True, regex=True)
    df['choice'].replace(x, y, inplace=True, regex=True)

update_size_list = [['16.7', '.05'],
                   [ '66.6','2'] ,
                   ['33.3','1']]
for x,y in update_size_list:
    df['size'].replace(x, y, inplace=True, regex=True)
# df['size'].unique()
    
df.rename(columns={"age": "age_original"}, inplace=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [12]:
haun2011great_standardized=df[['study_id','year','month', 'day', 
 'participant', 'age_original','age_in_years', 'sex', 'species', 'session',
       'trial',   'condition_codes', 'size', 'first_codes', 'no_of_cups', 'what_cups',
       'baited_cup', 'choice','success_codes','risk_choice_codes', 'comments']]

haun2011great_standardized.columns = haun2011great_standardized.columns.str.replace('_codes', '', regex=True)



In [ ]:




comp_out_path_stand = os.path.join(out_pathway, 'haun2011great_standardized.csv')
haun2011great_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =haun2011great_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
haun2011great_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'haun2011great_glossary.csv')
haun2011great_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
